# Ordered Logistic Regression Results: FAIRⁿ² Dataset Exploration with `mlcroissant`
This notebook demonstrates loading and exploring the [FAIRⁿ²](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json) dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library, following the Croissant standard.

### Dataset Source
The dataset's metadata and schema are defined using [Croissant](https://mlcommons.org/croissant/) at the following URL:

[https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json)

---

In [ ]:
# Install the mlcroissant library if not already installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load dataset metadata and available recordsets using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"\033[1m{metadata.name}\033[0m, version {metadata.version}")
print(f"\nDescription:\n{metadata.description}\n\n")
print("License:", metadata.license)
print("Citation:", metadata.citeAs)

## 2. Data Overview
Inspect the dataset structure:
- Record sets and their `@id`s
- Fields (columns) for each record set

**All references use `@id` values as required by Croissant.**

In [ ]:
# List all record set @ids and their columns
record_sets = list(dataset.record_sets.keys())
print(f"Record sets available (by @id):\n{record_sets}\n")

# For each record set, show corresponding fields (@id)
for record_set_id in record_sets:
    rs = dataset.record_sets[record_set_id]
    print(f"\n\u2022 Record set @id: {record_set_id}")
    if hasattr(rs, 'fields') and len(rs.fields) > 0:
        print("  Fields (@id):")
        for field in rs.fields:
            print(f"    - {field['@id']} ({field.get('name', '<no name>')})")
    else:
        print("  [No fields found]")

## 3. Data Extraction
Load tabular data from the relevant record sets into pandas DataFrames.

**Note:** Use `@id` values exactly as listed above for referencing.

In this dataset, the main record set appears to be where the regression results are stored. We'll attempt to load all detected recordsets.

In [ ]:
# Extract data from all record sets (using @ids)
dataframes = {}
failed = []

for record_set_id in record_sets:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if len(records) > 0:
            dataframes[record_set_id] = pd.DataFrame(records)
            print(f"Loaded {len(records)} records from record set {record_set_id}")
        else:
            print(f"No records found for record set {record_set_id}")
    except Exception as e:
        print(f"Failed for {record_set_id}: {e}")
        failed.append(record_set_id)

# Print available DataFrame(s) columns
if dataframes:
    # Select the largest DataFrame (most columns) as main df
    main_record_set_id = max(dataframes, key=lambda k: len(dataframes[k].columns))
    print(f"\nMain record set for further analysis: {main_record_set_id}")
    print(f"Columns (@id) in this record set:")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No dataframes loaded. Check the dataset definition and record sets above.")

## 4. Exploratory Data Analysis (EDA)
Let's process and explore the main record set. We'll:
- Filter the data by a numerical field (e.g., log likelihood, coefficient, or similar)
- Normalize a field
- Group by a categorical variable (e.g., knowledge source or region)

**Note:** Update the `numeric_field_id` and `group_field_id` based on the record set structure found above (by `@id`).

In [ ]:
# --- Specify the correct column @ids below after inspecting columns ---
df = dataframes[main_record_set_id]

# Try to programmatically identify possible numeric fields for demo (e.g., coefficient values, loglikelihood)
possible_numeric_fields = [col for col in df.columns if df[col].dtype.kind in 'ifc']
print("Numeric columns (by @id):", possible_numeric_fields)

# Example: select a numeric field for demonstration
if possible_numeric_fields:
    numeric_field_id = possible_numeric_fields[0]  # Select first found
else:
    raise RuntimeError("No numeric columns detected in the record set.")

print(f"\nUsing numeric field: {numeric_field_id}")

# If unable to auto-detect, set manually by inspecting the df.columns
# numeric_field_id = '<update with exact @id>'

threshold = df[numeric_field_id].mean() if pd.notna(df[numeric_field_id].mean()) else 0
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold:.3f}:")
display(filtered_df.head())

# Normalize the numeric field
filtered_df[numeric_field_id + "_normalized"] = (
    (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
)
print(f"\nNormalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, numeric_field_id + "_normalized"]].head())

# Try to find a group/categorical field
possible_group_fields = [col for col in df.columns if df[col].dtype == object]
if possible_group_fields:
    group_field_id = possible_group_fields[0]  # Take the first object column
    print(f"\nGrouping by: {group_field_id}")
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame('mean_' + numeric_field_id)
    print(f"\nMean {numeric_field_id} by group ({group_field_id}):")
    display(grouped_df.head())
else:
    print("\nNo suitable object/categorical column found for grouping.")

## 5. Visualization
We'll visualize the distribution of the selected numeric field and its relation to the group field (if found).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

plt.figure(figsize=(8, 4))
sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=20)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.show()

# Visualize by group if grouping field exists
if 'group_field_id' in locals():
    plt.figure(figsize=(10, 5))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=filtered_df)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
- The FAIRⁿ² dataset for ordered logistic regression in rangeland knowledge adoption is accessible via Croissant and easily parsed with `mlcroissant`.
- Structure and schema, including record sets and fields, are referenced by their `@id` per Croissant conventions.
- This notebook demonstrated programmatic metadata inspection, flexible tabular extraction, numeric filtering, normalization, grouping, and quick visualization to enable further analytical workflows.

_For deeper insights, consult the full documentation and schema at the dataset's Croissant URL._